<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - Planted area Processor Bulk Extraction

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 -Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from processor_planted_functions.py**

### 🗺️ Configure extraction

In [ ]:
# Import your class
from earthdaily.agriculture.processors.processor_plantedarea_functions import PlantedExtractor
extractor = PlantedExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )
# Setup parameters for PLANTED_AREA mode
extractor.setup_planted_parameters(
    processor_mode="PLANTED_AREA",        # Options: "PLANTED_AREA" or "CONTROL"
    emergence_date="2025-04-01",          # Required: YYYY-MM-DD format
    threshold=30,                        # Days between images (before emergence and second image)
    control_threshold=4,                  # Percentage threshold (only used in CONTROL mode)
    publish_af=False,                     # Whether to publish to AF database
    partial_frequency=50,                 # How often to save partial results
)

extractor2 = PlantedExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )
#Setup for CONTROL mode
extractor2.setup_planted_parameters(
    processor_mode="CONTROL",
    emergence_date="2025-04-01",
    threshold=120,
    control_threshold=4,                # Will compare planted area % against this threshold
    publish_af=False,
    partial_frequency=50,
)

### 🗺️ Test functions

In [ ]:
# Prepare test seasonfield_data
seasonfield_data = {
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "OTHERS",
    "emergence_date": "2025-04-02"
}

#### Test get_planted_api

In [ ]:
print("\n--- Test: get_planted ---")
# Invoke the function
try:
    result = extractor.get_planted_api(seasonfield_data)
    print("✅ Raw API response received:")
    print(result if isinstance(result, dict) else result[:500])
except Exception as e:
    print(f"❌ API call failed: {e}")
    


#### Test get_planted_api_safe

In [ ]:
print("\n--- Test: get_emergence_safe ---")
safe_result = extractor.get_planted_api_safe(seasonfield_data)
print(safe_result)

#### Test format_planted_json

In [ ]:
print("\n--- Test: format_planted_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_planted_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
else:
    print("⚠️ Skipping format_planted_json: No valid data from API.")

### 🗺️ process_single_entity_planted

In [ ]:
import pandas as pd
row = pd.Series({
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop":"OTHERS",
    "emergence_date": "2025-04-02"
})

result = extractor.process_single_entity_planted(row)

print(result)

### 🗺️ Per-entity emergence_date (wired from EmergenceExtractor output)

`setup_planted_parameters` no longer *requires* `emergence_date` — each entity can carry its own. Two patterns:

1. **Canonical column**: the row has an `emergence_date` field (e.g. YYYY-MM-DD string or `pd.Timestamp`). It's auto-normalized to YYYY-MM-DD by `BaseExtractor`.
2. **Upstream output with `column_mapping`**: `EmergenceExtractor` returns a different column name (e.g. `detected_emergence_date`). Map it to the canonical `emergence_date` via `column_mapping`.

If both row and params supply a value, the **row wins**. If neither is present, the entity is rejected with a clear error.

In [ ]:
# Scenario A — default in setup, but the row carries its own emergence_date (row wins)
extractor_per_entity = PlantedExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)
extractor_per_entity.setup_planted_parameters(
    processor_mode='PLANTED_AREA',
    # No emergence_date here — every row must provide one.
)

row_with_emergence = pd.Series({
    'id': 'z361x33',
    'geometry': 'POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))',
    'crop': 'OTHERS',
    'emergence_date': '2025-04-10',  # per-entity override
})

result_row = extractor_per_entity.process_single_entity_planted(row_with_emergence)
print(result_row)

#### Scenario B — wire `EmergenceExtractor` output via `column_mapping`

Say the upstream emergence DataFrame uses a column called `detected_emergence_date`. Map it:

In [ ]:
# Simulate the shape of an upstream extractor output (EmergenceExtractor-like)
emergence_output = pd.DataFrame([
    {
        'entity_id': 'z361x33',
        'detected_emergence_date': '2025-04-12',
        'geometry': 'POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))',
        'crop': 'OTHERS',
    }
])

extractor_mapped = PlantedExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)
extractor_mapped.setup_planted_parameters(
    processor_mode='PLANTED_AREA',
    column_mapping={
        'id': 'entity_id',                        # upstream calls it entity_id
        'emergence_date': 'detected_emergence_date',  # map upstream col → canonical
    },
)

result_bulk = extractor_mapped.process_planted_bulk_extraction_parallel(
    entity_list=emergence_output,
    max_workers=2,
    output_path=manager.output_result_dir,
    prefix='planted_from_emergence',
    skip_export=True,
)
print(f"Success: {result_bulk['successful_calculations']}/{result_bulk['total_calculations']}")
if not result_bulk['results_df'].empty:
    display(result_bulk['results_df'])

### 🗺️ process_planted_bulk_extraction_parallel

In [ ]:

top25 = manager.sfd_list.head(50)
top25=top25.rename(columns={"crop.id": "crop"})
print(top25.columns)
# Launch extraction with 10 threads 

result = extractor.process_planted_bulk_extraction_parallel(
    entity_list=top25,
    params=None,
    max_workers=20,
    output_path=manager.output_result_dir,
    fail_safe=False,
    filter_column="crop",
    filter_value="OTHERS",
    filter_type="include" # filter type used to 'include' or 'exclude' row matching column and value filter
)

print(f"\nResults summary:")
print(f"Total: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")
print(result["results_df"].columns)
print("\n🔍 First 3 errors:")
for i, error in enumerate(result['global_errors'][:3]):
    print(f"\nError {i+1}:")
    for key, value in error.items():
        print(f"  {key}: {value}")

In [ ]:
print(result["results_df"])

In [ ]:
# Get the clean DataFrame
results=result["results_df"]
print(results.columns)

In [ ]:
print(manager.output_result_dir)